# Assignment 14: Feature Engineering, Encoding, Scaling & Pipelines

### Task 1: Creating New Features

In [ ]:
import pandas as pd

df = pd.read_csv('employee_data.csv')

df['projects_per_year'] = df['projects_completed'] / df['experience']
df['age_group'] = pd.cut(df['age'], bins=[20, 30, 40, 50], labels=['Young', 'Mid', 'Senior'])
df[['age', 'age_group', 'experience', 'projects_per_year']].head()

### Task 2: Handling Date Features

In [ ]:
df['joining_date'] = pd.to_datetime(df['joining_date'])
df['year'] = df['joining_date'].dt.year
df['month'] = df['joining_date'].dt.month
df['day'] = df['joining_date'].dt.day
df[['joining_date', 'year', 'month', 'day']].head()

### Task 3: One-Hot Encoding

In [ ]:
encoded_df = pd.get_dummies(df, columns=['department'], drop_first=True)
encoded_df.head()

### Task 4: ColumnTransformer

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

num_cols = ['age', 'experience', 'projects_completed']
cat_cols = ['department']

ct = ColumnTransformer([
    ('ohe', OneHotEncoder(drop='first', sparse_output=False), cat_cols)
], remainder='passthrough')

data = ct.fit_transform(df[num_cols + cat_cols])
data[:3]

### Task 5: StandardScaler

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaled = pd.DataFrame(scaler.fit_transform(df[num_cols]), columns=num_cols)
print('Mean (~0):\n', scaled.mean().round(2))
print('Std (~1):\n', scaled.std().round(2))
scaled.head()

### Task 6: MinMaxScaler

In [ ]:
from sklearn.preprocessing import MinMaxScaler

minmax = MinMaxScaler().fit_transform(df[num_cols])
minmax[:3].round(2)

### Task 7: Preprocessing Pipeline

In [ ]:
from sklearn.pipeline import Pipeline

num_pipe = Pipeline([('scale', StandardScaler())])
cat_pipe = Pipeline([('ohe', OneHotEncoder(drop='first', sparse_output=False))])

preprocessor = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols)
])

preprocessor.fit_transform(df[num_cols + cat_cols])[:3].round(2)

### Task 8: Full Pipeline

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

X = df[num_cols + cat_cols]
y = df['salary']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipeline = Pipeline([
    ('prep', preprocessor),
    ('model', LinearRegression())
])

pipeline.fit(X_train, y_train)
print('Score:', round(pipeline.score(X_test, y_test), 3))

### Task 9: Pipeline Benefits

1. **Why pipelines are important**: Chains preprocessing and modeling in one object.
2. **Problems solved**: Prevents data leakage and avoids repeating code on test data.
3. **Manual vs Pipeline**: Manual preprocessing transforms data separately with many steps, while pipelines do it automatically.